In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

config_str = os.getenv("GPT4O_MINI_CONFIG")

if config_str is None:
    raise ValueError("model environment variable is missing")

config = json.loads(config_str)
MODEL_NAME = config["name"]
REASONING_EFFORT = config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

SLEEP_BETWEEN_CALLS = int(
    os.getenv("SLEEP_BETWEEN_CALLS", "1")
)


client = OpenAI(
    api_key=OPENAI_API_KEY
)

VALID_CATEGORIES = {
    "Correctness",
    "Design",
    "Maintainability",
    "Readability",
    "Documentation",
}

DATA_PATH = os.getenv(
    "DATA_PATH"
)

df=pd.read_csv(f'{DATA_PATH}df_n2.csv')

Using model: gpt-4o-mini-2024-07-18 with reasoning effort: 


In [ ]:

import json
import time
import uuid
import random



# ==========================
# Global token usage
# ==========================

TOTAL_USAGE = {
    "prompt_tokens": 0,
    "reasoning_tokens": 0,
    "total_tokens": 0,
    "num_calls": 0,
}

def update_usage(usage):
    """
    Accumulate token usage across all LLM calls.
    """
    global TOTAL_USAGE

    u = usage.model_dump()

    TOTAL_USAGE["prompt_tokens"] += u.get("prompt_tokens", 0)
    TOTAL_USAGE["total_tokens"] += u.get("total_tokens", 0)
    TOTAL_USAGE["reasoning_tokens"] += (
        u.get("completion_tokens_details", {})
         .get("reasoning_tokens", 0)
    )
    TOTAL_USAGE["num_calls"] += 1
     
     

def build_messages(row):

    categories_text = "\n".join(
        f"- {category}"
        for category in VALID_CATEGORIES
    )
    system_prompt = f"""
You are an expert software engineer performing a professional code review.

INPUT:
1. Pull request context (when available)
2. Surrounding code context (when available), wrapped in <context> tags
3. The code patch (git diff), wrapped in <patch> tags

TASK:
Identify review comments that a human reviewer would reasonably leave on the changes in <patch>. For each category below, use the available context when it helps determine whether the patch introduces an issue or an opportunity for improvement:

- CORRECTNESS: Does the patch introduce an error, undefined reference, or unexpected behavior? Does a changed name/parameter/value conflict with, or fail to match, how it's used elsewhere in <context>? Point to the specific location if so.
- MAINTAINABILITY: Does the patch duplicate, remove, or diverge from logic that exists in a structurally similar place in <context> (sibling method, sibling class, repeated pattern)? Is there hard-coded or fragile logic that will make future changes harder? Prefer suggesting consolidation over just noting duplication.
- DESIGN: Is there a simpler, more consistent architecture or implementation pattern that fits how similar problems are solved elsewhere in <context>?
- READABILITY: Is the code harder to understand than it needs to be, independent of whether it's correct — unclear naming, formatting, or control flow?
- DOCUMENTATION: Is a docstring or comment now inaccurate, missing, or inconsistent with the code's actual behavior? If a name/value conflicts with what's documented elsewhere in <context> without affecting runtime behavior, treat it as Documentation rather than Correctness.

RULES:
- Only comment on things caused by or directly connected to <patch>. Don't flag unrelated pre-existing issues.
- Don't hallucinate code that isn't shown.
- One issue per comment, and each comment belongs to exactly one category. Be concise and specific — point at the exact location/identifier rather than restating the diff.
- It's fine to phrase a comment as a direct question when that's how a reviewer would naturally raise it.
- Before assigning a category, briefly justify why that category fits over the others it could plausibly be confused with (e.g. Design vs. Readability, Correctness vs. Documentation).

Categories:
{VALID_CATEGORIES}

OUTPUT (STRICT JSON ONLY, no other text):
{{"comments": [{{"comment": "...", "category_justification": "...", "category": "...", "severity": "Low | Medium | High"}}]}}

If nothing meaningful: {{"comments": []}}
"""


    user_prompt = ""



    if "pr_title" in row and pd.notna(row["pr_title"]):

        user_prompt += f"""
PULL REQUEST TITLE:
{row["pr_title"]}

"""



    if "target_file" in row and pd.notna(row["target_file"]):

        user_prompt += f"""
TARGET FILE:
{row["target_file"]}

"""
    if "relevant_same_file_code_hunks" in row and pd.notna(row["relevant_same_file_code_hunks"]):

        user_prompt += f"""
RELATED CODE HUNKS IN TARGET FILE:
{row["relevant_same_file_code_hunks"]}

"""

    if "changed_files" in row and pd.notna(row["changed_files"]):

        user_prompt += f"""
FILES CHANGED:
{row["changed_files"]}

"""

    if "relevant_different_files_code_hunks" in row and pd.notna(row["relevant_different_files_code_hunks"]):

        user_prompt += f"""
RELATED CODE HUNKS IN DIFFERENT FILE:
{row["relevant_different_files_code_hunks"]}

"""

    if "relevant_context" in row and pd.notna(row["relevant_context"]):

        if str(row["relevant_context"]).strip():

            user_prompt += """
SURROUNDING CODE CONTEXT:
This is additional context extracted from the original file around the changed code to help you in problem identification.
<context>
"""

            user_prompt += row["relevant_context"]
            user_prompt += "\n</context>\n\n"

    user_prompt += f"""
CODE PATCH:
<patch>
{row["hunk"]}
</patch>
"""

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        },
    ]
    
# ==========================
# Output parsing
# ==========================

def parse_output(text, code_diff_idx):

    if text is None:
        return []

    try:
        data = json.loads(text)

        comments = data.get("comments", [])

        parsed = []

        for j, c in enumerate(comments):

            category = str(c.get("category", "")).strip()
            comment = str(c.get("comment", "")).strip()
            severity = str(c.get("severity", "Medium")).strip()

            if category not in VALID_CATEGORIES:
                continue

            if len(comment) == 0:
                continue

            parsed.append({
                "category": category,
                "comment": comment,
                "severity": severity,
            })

        return parsed

    except Exception:
        return []



# ==========================
# OpenAI prediction
# ==========================

def predict_once(row):

    messages = build_messages(row)

    
    kwargs = {
        "model": MODEL_NAME,
        "messages": messages,
        "timeout": 120,
    }

    if REASONING_EFFORT:
        kwargs["reasoning_effort"] = REASONING_EFFORT

    response = client.chat.completions.create(**kwargs)

    raw_text = (
        response
        .choices[0]
        .message
        .content
    )

    comments = parse_output(
        raw_text,
        row["idx"] if "idx" in row else None
    )


    # Add metadata to each generated comment
    for comment in comments:
        comment["id"] = str(uuid.uuid4())
        comment["generation_system"] = MODEL_NAME

    update_usage(response.usage) 
    return {
        "generated_comments": comments
    }



# ==========================
# Pipeline execution
# ==========================

def run_pipeline(df):
    

    preds = []

    total = len(df)

    processed = 0

    base_sleep = (
        SLEEP_BETWEEN_CALLS
        if "SLEEP_BETWEEN_CALLS" in globals()
        else 1.5
    )


    print(
        f"[START] Processing {total} rows"
    )


    for _, row in df.iterrows():

        processed += 1

        print(
            f"\n[ROW {processed}/{total}] Starting"
        )


        if pd.isna(row["hunk"]):

            print(
                "[SKIP] Missing data"
            )

            preds.append(
                {
                    "generated_comments": []
                }
            )

            continue


        success = False

        sleep_time = base_sleep


        for attempt in range(3):

            try:

                pred = predict_once(row)

                preds.append(
                    pred
                )

                success = True

                break


            except Exception as e:

                print(
                    f"[ERROR] {e}"
                )

                wait = (
                    sleep_time
                    + random.uniform(0, 1)
                )

                print(
                    f"[RETRY] waiting {wait:.2f}s"
                )

                time.sleep(wait)

                sleep_time *= 2



        if not success:

            print(
                "[FAIL] row exhausted retries"
            )

            preds.append(
                {
                    "generated_comments": []
                }
            )


        wait = (
            base_sleep
            + random.uniform(0, 0.8)
        )

        print(
            f"[SLEEP] pacing before next request: {wait:.2f}s"
        )

        time.sleep(wait)


        if processed % 10 == 0:

            print(
                f"[CHECKPOINT] processed {processed}/{total}"
            )


            backup_df = pd.concat(
                [
                    df.iloc[:processed].reset_index(drop=True),
                    pd.DataFrame(preds),
                ],
                axis=1,
            )


            backup_df.to_csv(
                "artifacts/backup_predictions.csv",
                index=False
            )

            print(
                "[CHECKPOINT] saved backup_predictions.csv"
            )



    print(
        "\n[DONE] Building dataframe"
    )


    pred_df = pd.DataFrame(
        preds
    )

    df = df.reset_index(
        drop=True
    )


    print(
        "[DONE] Finished pipeline"
    )

    return pd.concat(
        [
            df,
            pred_df
        ],
        axis=1
    )

In [12]:
import pandas as pd

def create_generated_comments_dataset(df):
    rows = []

    for _, row in df.iterrows():
        comments = row["generated_comments"]

        # Skip empty/null comments
        if not isinstance(comments, list):
            continue

        for comment in comments:
            new_row = {
                "comment_id": comment.get("id"),
                "comment": comment.get("comment"),
                "category": comment.get("category"),
                "generation_system": comment.get("generation_system"),
                "patch_id": row["patch_id"],
                "severity": comment.get("severity"),
                # "github_commit_url": row["github_commit_url"],
                # "pr_url": row["pr_url"]
            }

            rows.append(new_row)

    return pd.DataFrame(rows)

In [ ]:
df_results= run_pipeline(df)
df_generated_comments= create_generated_comments_dataset(df_results)
df_generated_comments.to_csv(f'{DATA_PATH}df_n3_3.csv', index=False)

[START] Processing 10 rows

[ROW 1/10] Starting
[SLEEP] pacing before next request: 1.52s

[ROW 2/10] Starting
[SLEEP] pacing before next request: 1.50s

[ROW 3/10] Starting
[SLEEP] pacing before next request: 1.35s

[ROW 4/10] Starting
[SLEEP] pacing before next request: 1.79s

[ROW 5/10] Starting
[SLEEP] pacing before next request: 1.61s

[ROW 6/10] Starting
[SLEEP] pacing before next request: 1.24s

[ROW 7/10] Starting
[SLEEP] pacing before next request: 1.61s

[ROW 8/10] Starting
[SLEEP] pacing before next request: 1.28s

[ROW 9/10] Starting
[SLEEP] pacing before next request: 1.51s

[ROW 10/10] Starting
[SLEEP] pacing before next request: 1.25s
[CHECKPOINT] processed 10/10
[CHECKPOINT] saved backup_predictions.csv

[DONE] Building dataframe
[DONE] Finished pipeline

========== LLM USAGE SUMMARY ==========
Number of LLM calls : 11
Prompt tokens       : 48,206
Reasoning tokens    : 0
Total tokens        : 49,832



In [ ]:
import json
from datetime import datetime
from pathlib import Path


def save_token_usage_log(
    task_name="comment_generation",
    log_file="../logs/token_usage_insights_logs.json"
):

    usage_stats = {
        "task": task_name,
        "timestamp": datetime.now().isoformat(),
        "model_name": MODEL_NAME,
        "num_calls": TOTAL_USAGE["num_calls"],
        "prompt_tokens": TOTAL_USAGE["prompt_tokens"],
        "reasoning_tokens": TOTAL_USAGE["reasoning_tokens"],
        "total_tokens": TOTAL_USAGE["total_tokens"],
    }

    log_path = Path(log_file)

    log_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        with open(log_path, "r") as f:
            logs = json.load(f)

    except (FileNotFoundError, json.JSONDecodeError):
        logs = []

    logs.append(usage_stats)

    with open(log_path, "w") as f:
        json.dump(
            logs,
            f,
            indent=4
        )

    print(f"Token usage saved to {log_path}")
    
save_token_usage_log()    